# KUKA KRL Path Visualizer
**Author:** Thejas Dixit Sathyanarayana  
**GitHub:** [Thejas12Dixit](https://github.com/Thejas12Dixit)  
**Description:** Parse KUKA KRL `.src` / `.dat` files and visualize robot paths interactively in 3D. Export to PDF or PNG for documentation.

---

## 1. Setup

In [ ]:
%matplotlib notebook
import sys, os
sys.path.insert(0, os.getcwd())

from krl_parser import KRLParser, KRLVisualizer
import matplotlib.pyplot as plt

print('✔ Modules loaded')

## 2. Parse KRL files
Point the parser at your `.src` file. The `.dat` is auto-detected if in the same folder.

In [ ]:
# ── Configure paths ──────────────────────────
SRC_FILE = 'sample_welding.src'   # Change to your file
DAT_FILE = None                   # Set to path or leave None for auto-detect
# ─────────────────────────────────────────────

parser  = KRLParser()
program = parser.parse(SRC_FILE, DAT_FILE)
vis     = KRLVisualizer(program)

if program.warnings:
    print('⚠ Warnings:')
    for w in program.warnings:
        print(f'  {w}')
else:
    print(f'✔ Parsed successfully — {len(program.motions)} motion commands found')

## 3. Program statistics

In [ ]:
stats = vis.get_stats()

print('─' * 40)
print(f"  Program       : {stats['program_name']}")
print(f"  Total points  : {stats['total_points']}")
print(f"  PTP moves     : {stats['ptp_moves']}")
print(f"  LIN moves     : {stats['lin_moves']}")
print(f"  CIRC moves    : {stats['circ_moves']}")
print(f"  Total distance: {stats['total_distance']} mm")
print('─' * 40)
print('  Workspace envelope:')
print(f"  X: {stats['x_range'][0]} → {stats['x_range'][1]} mm")
print(f"  Y: {stats['y_range'][0]} → {stats['y_range'][1]} mm")
print(f"  Z: {stats['z_range'][0]} → {stats['z_range'][1]} mm")
print('─' * 40)
print(f"  Velocity range: {stats['min_velocity_mms']} → {stats['max_velocity_mms']} mm/s (LIN/CIRC)")

## 4. Interactive 3D path plot
Rotate with mouse · Zoom with scroll

In [ ]:
fig, ax = vis.plot_3d()
plt.tight_layout()
plt.show()

## 5. Motion sequence table

In [ ]:
import pandas as pd

rows = []
for i, motion in enumerate(program.motions):
    if motion.point is None:
        continue
    pt = motion.point
    rows.append({
        '#':          i + 1,
        'Type':       motion.motion_type,
        'Point':      motion.point_name,
        'X (mm)':     round(pt.x, 1),
        'Y (mm)':     round(pt.y, 1),
        'Z (mm)':     round(pt.z, 1),
        'A (deg)':    round(pt.a, 1),
        'B (deg)':    round(pt.b, 1),
        'C (deg)':    round(pt.c, 1),
        'Velocity':   f"{motion.velocity} {motion.velocity_unit}" if motion.velocity else '—',
    })

df = pd.DataFrame(rows)

def color_type(val):
    colors = {'PTP': '#D6EAF8', 'LIN': '#D5F5E3', 'CIRC': '#FDEBD0'}
    return f'background-color: {colors.get(val, "white")}'

df.style.applymap(color_type, subset=['Type'])

## 6. Export to PDF and PNG

In [ ]:
# Export PDF report (2 pages: 3D plot + motion table)
pdf_path = vis.export_pdf(f'{program.name}_report.pdf')
print(f'✔ PDF saved → {pdf_path}')

# Export PNG
png_path = vis.export_png(f'{program.name}_path.png')
print(f'✔ PNG saved → {png_path}')

## 7. Try your own file
Replace `SRC_FILE` with the path to your actual KUKA program and re-run from cell 2.